# 11 — Phase 4-E: 3계층 XAI 통합 파이프라인

> **목표:** Grad-CAM + 수치 앵커링 + LLaVA 캡션을 하나의 `explain()` 함수로 통합  
> **입력:** 이미지 1장 (`img_bgr`)  
> **출력:**
> - `verdict` : REAL / FAKE
> - `spoof_prob` : 판정 확률 (0~1)
> - `spoof_type` : 예측된 공격 유형
> - `heatmap_overlay` : Grad-CAM 히트맵 오버레이 이미지
> - `anchor_stats` : 활성 영역 Laplacian / FFT 수치
> - `xai_text` : 자연어 설명 텍스트 (Layer 3)
>
> **산출물:** `src/xai_explainer.py` — Streamlit에서 `from xai_explainer import explain` 으로 import

## Cell 0 — Google Drive 마운트 (항상 먼저 실행)

In [1]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive 마운트 완료')

Mounted at /content/drive
✅ Drive 마운트 완료


## Cell 1 — 라이브러리 & 경로 설정

In [24]:
import os, json, re
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import tensorflow as tf
from pathlib import Path
import random

# ── 경로 ──────────────────────────────────────────────────
BASE        = '/content/drive/MyDrive/face-anti-spoofing'
MODEL_DIR   = f'{BASE}/models'
CROP_DIR    = f'{BASE}/data/cropped'
REPORT_DIR  = f'{BASE}/reports/phase4'
CAPTION_JSON= f'{BASE}/results/phase4/llava_captions.json'
SRC_DIR     = f'{BASE}/src'
os.makedirs(REPORT_DIR, exist_ok=True)
os.makedirs(SRC_DIR,    exist_ok=True)

print('TF  :', tf.__version__)
print('GPU :', tf.config.list_physical_devices('GPU'))
print('BASE:', BASE)

TF  : 2.20.0
GPU : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
BASE: /content/drive/MyDrive/face-anti-spoofing


## Cell 2 — 모델 로드 & 레이어 자동 감지

In [3]:
model = tf.keras.models.load_model(f'{MODEL_DIR}/stage2_best.h5')
print('✅ 모델 로드 완료')

# ── Grad-CAM 대상 Conv 레이어 자동 감지 ───────────────────
conv_layers = [l.name for l in model.layers
               if 'conv' in l.name.lower() and hasattr(l, 'filters')]
TARGET_LAYER = conv_layers[-1]
print(f'  Grad-CAM 타겟: {TARGET_LAYER}')

# ── sigmoid 직전 logit 레이어 자동 감지 ───────────────────
logit_layer_name = None
prev_name = None
for layer in model.layers:
    cfg = layer.get_config()
    act = cfg.get('activation', '')
    if isinstance(act, dict):
        act = act.get('class_name', '')
    if act == 'sigmoid' and 'dense' in layer.name.lower():
        logit_layer_name = prev_name
        print(f'  sigmoid 레이어: {layer.name}')
        print(f'  logit 후보: {logit_layer_name}')
        break
    prev_name = layer.name

# fallback: binary dense 직접 탐색
if logit_layer_name is None:
    for layer in model.layers:
        if 'binary' in layer.name.lower() and 'dense' in layer.name.lower():
            logit_layer_name = layer.name
            print(f'  binary dense 직접 감지: {logit_layer_name}')
            break

LOGIT_LAYER = logit_layer_name or 'head_binary'
print(f'\n✅ LOGIT_LAYER: {LOGIT_LAYER}')

# ── Spoof Type 출력 헤드 확인 ──────────────────────────────
SPOOF_LAYER = None
for layer in model.layers:
    if 'spoof' in layer.name.lower() and 'dense' in layer.name.lower():
        SPOOF_LAYER = layer.name
        print(f'✅ SPOOF_LAYER: {SPOOF_LAYER}')
        break
if SPOOF_LAYER is None:
    print('⚠️ spoof head 자동 감지 실패 → Cell 아래에서 수동 지정')

# 수동 지정 (자동 감지 실패 시 아래 주석 해제 후 레이어 이름 입력)
# LOGIT_LAYER = 'head_binary'
# SPOOF_LAYER = 'head_spoof'

✅ 모델 로드 완료
  Grad-CAM 타겟: Conv_1

✅ LOGIT_LAYER: head_binary
⚠️ spoof head 자동 감지 실패 → Cell 아래에서 수동 지정


In [5]:
# 수동 지정 (자동 감지 실패 시)
LOGIT_LAYER = 'binary'   # ← 이미 맞음
SPOOF_LAYER = 'spoof'    # ← 이걸 추가
TARGET_LAYER = 'Conv_1'  # ← 이미 맞음

print(f'✅ TARGET_LAYER : {TARGET_LAYER}')
print(f'✅ LOGIT_LAYER  : {LOGIT_LAYER}')
print(f'✅ SPOOF_LAYER  : {SPOOF_LAYER}')

✅ TARGET_LAYER : Conv_1
✅ LOGIT_LAYER  : binary
✅ SPOOF_LAYER  : spoof


## Cell 3 — LLaVA 캡션 사전 로드

> `llava_captions.json` 에서 이미지 경로 → 캡션 딕셔너리를 메모리에 올려둔다.  
> 캡션이 없는 이미지는 fallback 문자열 사용.

In [19]:
# Cell 3 교체 — subset 경로 기반으로 CAPTION_DB 재구성 (기존과 동일)
# + 카테고리별 캡션 풀도 따로 만들기 (cropped 파일과 매핑 불가할 때 fallback용)

CAPTION_DB = {}
CAPTION_POOL = {'live': [], 'print': [], 'replay': [], 'mask': []}

if Path(CAPTION_JSON).exists():
    with open(CAPTION_JSON) as f:
        records = json.load(f)
    for rec in records['results']:
        stem = Path(rec['img_path']).stem
        caption = rec.get('caption', '')
        CAPTION_DB[stem] = caption
        cat = rec.get('category', '')
        if cat in CAPTION_POOL and caption:
            CAPTION_POOL[cat].append(caption)
    print(f"✅ 캡션 DB 로드: {len(CAPTION_DB)}개")
    for cat, pool in CAPTION_POOL.items():
        print(f"  [{cat}] 풀 {len(pool)}개")
else:
    print(f"⚠️ {CAPTION_JSON} 없음")

✅ 캡션 DB 로드: 200개
  [live] 풀 50개
  [print] 풀 50개
  [replay] 풀 50개
  [mask] 풀 50개


## Cell 4 — Layer 1: Logit 기반 Grad-CAM 함수

> 07번 노트북에서 완성된 함수를 그대로 가져온다.

In [20]:
def preprocess(img_bgr, size=224):
    """BGR 이미지 → 모델 입력 tensor (1, 224, 224, 3)"""
    img_rgb  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_res  = cv2.resize(img_rgb, (size, size))
    img_norm = img_res.astype('float32') / 255.0
    return np.expand_dims(img_norm, 0)


def get_gradcam_logit(model, img_array, conv_layer_name, logit_layer_name):
    """
    [Layer 1] logit 기반 Grad-CAM
    반환: heatmap (H×W, 0~1 정규화)
    """
    try:
        logit_output = model.get_layer(logit_layer_name).output
    except ValueError:
        print(f'⚠️ {logit_layer_name} 레이어 없음')
        return np.zeros((7, 7))

    grad_model = tf.keras.Model(
        inputs =model.inputs,
        outputs=[model.get_layer(conv_layer_name).output, logit_output]
    )
    with tf.GradientTape() as tape:
        inp      = tf.cast(img_array, tf.float32)
        conv_out, pred = grad_model(inp)
        # sigmoid 이미 적용된 경우 → logit 역변환
        p_clipped = tf.clip_by_value(pred[:, 0], 1e-7, 1 - 1e-7)
        logit     = tf.math.log(p_clipped / (1.0 - p_clipped))
        loss      = logit

    grads  = tape.gradient(loss, conv_out)
    if grads is None:
        print('⚠️ gradient = None')
        return np.zeros((7, 7))

    pooled  = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = conv_out[0] @ pooled[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0)
    heatmap = heatmap / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()


def overlay_heatmap(img_bgr, heatmap, alpha=0.45):
    """원본 이미지에 히트맵 오버레이 → RGB ndarray"""
    h, w   = img_bgr.shape[:2]
    hmr    = cv2.resize(heatmap, (w, h))
    hmc    = cv2.applyColorMap(np.uint8(255 * hmr), cv2.COLORMAP_JET)
    img_r  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    hmc_r  = cv2.cvtColor(hmc,     cv2.COLOR_BGR2RGB)
    return cv2.addWeighted(img_r, 1 - alpha, hmc_r, alpha, 0)


print('✅ Layer 1 (Grad-CAM) 함수 준비 완료')

✅ Layer 1 (Grad-CAM) 함수 준비 완료


## Cell 5 — Layer 2: 수치 앵커링 함수

> Grad-CAM 히트맵에서 **활성 영역 마스크**를 추출한 뒤,  
> 해당 영역에만 Laplacian / FFT 에너지를 재측정한다.

In [21]:
# ── 기준값 (Phase 4-B 실측, 카테고리 평균) ──────────────
ANCHOR_BASELINE = {
    'live'   : {'laplacian': 383, 'fft_high': 1134},
    'print'  : {'laplacian': 318, 'fft_high': 1042},
    'replay' : {'laplacian': 319, 'fft_high':  944},
    'mask'   : {'laplacian': 480, 'fft_high': 1134},
}


def get_active_mask(heatmap, threshold=0.4):
    """
    히트맵에서 활성 영역 마스크 추출
    threshold 이상인 픽셀만 True
    """
    return heatmap >= threshold


def compute_pixel_features(img_bgr, mask_224=None):
    """
    이미지 전체 또는 마스크 영역에 대해 Laplacian / FFT 계산
    mask_224: (224, 224) bool 배열. None이면 전체 이미지 사용
    반환: {'laplacian': float, 'fft_high': float, 'fft_low': float}
    """
    img_r   = cv2.resize(img_bgr, (224, 224))
    img_g   = cv2.cvtColor(img_r, cv2.COLOR_BGR2GRAY)  # uint8

    if mask_224 is not None:
        # 마스크 영역 픽셀만 추출
        pixels = img_g[mask_224].astype(np.float32)
        if len(pixels) < 16:
            mask_224 = None  # 픽셀 너무 적으면 전체로 fallback

    if mask_224 is not None:
        # Laplacian: 마스크 적용 후 분산
        lap     = cv2.Laplacian(img_g, cv2.CV_64F)
        lap_var = float(lap[mask_224].var())

        # FFT: 마스크 영역 서브이미지로 계산
        ys, xs  = np.where(mask_224)
        y0, y1  = int(ys.min()), int(ys.max()) + 1
        x0, x1  = int(xs.min()), int(xs.max()) + 1
        sub     = img_g[y0:y1, x0:x1].astype(np.float32)
    else:
        lap     = cv2.Laplacian(img_g, cv2.CV_64F)
        lap_var = float(lap.var())
        sub     = img_g.astype(np.float32)

    # FFT 에너지
    f         = np.fft.fftshift(np.fft.fft2(sub))
    mag       = np.abs(f)
    h, w      = mag.shape
    cy, cx    = h // 2, w // 2
    r         = min(h, w) // 6
    Y, X      = np.ogrid[:h, :w]
    d2        = (Y - cy)**2 + (X - cx)**2
    fft_high  = float(mag[d2 > r**2].mean())
    fft_low   = float(mag[d2 <= r].mean())

    return {'laplacian': round(lap_var, 1), 'fft_high': round(fft_high, 1), 'fft_low': round(fft_low, 1)}


def interpret_anchors(stats, category='unknown'):
    """수치를 바탕으로 해석 문자열 반환"""
    lines = []
    base  = ANCHOR_BASELINE.get(category, ANCHOR_BASELINE['live'])

    lap_diff = stats['laplacian'] - base['laplacian']
    if   lap_diff < -40:
        lines.append('질감 평탄 (선명도 낮음)')
    elif lap_diff >  40:
        lines.append('경계선 강조 (선명도 높음)')
    else:
        lines.append('선명도 보통')

    fft_diff = stats['fft_high'] - base['fft_high']
    if   fft_diff < -80:
        lines.append('고주파 에너지 부재 (압축/재촬영 흔적)')
    elif fft_diff >  80:
        lines.append('고주파 에너지 과잉 (경계 아티팩트)')
    else:
        lines.append('고주파 에너지 정상')

    return ' / '.join(lines)


print('✅ Layer 2 (수치 앵커링) 함수 준비 완료')

✅ Layer 2 (수치 앵커링) 함수 준비 완료


## Cell 6 — Layer 3: 자연어 설명 합성 함수

> Spoof Type 예측 + 수치 앵커링 결과 + LLaVA 캡션을 조합해  
> 최종 XAI 설명 텍스트를 만든다.

In [32]:
SPOOF_KO = {
    0: 'Live (실제 얼굴)',
    1: 'Print Attack (인쇄 공격)',
    2: 'Replay Attack (화면 재촬영)',
    3: '3D Mask (입체 마스크)',
}
SPOOF_EN_HINT = {
    1: '인쇄물 특유의 평탄한 피부 질감과 낮은 고주파 에너지가 관측됩니다.',
    2: '화면 재촬영 특유의 모아레 패턴 및 고주파 에너지 감소가 감지됩니다.',
    3: '마스크 경계부에 비정상적 선명도 및 피부 질감 불일치가 나타납니다.',
}

def get_llava_caption(img_path, category=None):
    """
    1순위: stem으로 직접 매칭 (subset 경로로 호출 시)
    2순위: 카테고리 풀에서 랜덤 (cropped 경로로 호출 시 → stem 불일치)
    """
    stem = Path(img_path).stem if img_path else ''

    # 직접 매칭
    if stem in CAPTION_DB:
        return CAPTION_DB[stem]

    # fallback: 카테고리 풀에서 랜덤
    if category and category in CAPTION_POOL and CAPTION_POOL[category]:
        return random.choice(CAPTION_POOL[category])

    return ''


def build_xai_text(verdict, spoof_prob, spoof_type_idx,
                   anchor_stats, anchor_interp,
                   llava_caption='', illum_idx=None, env_idx=None):
    """
    3계층 XAI 설명 텍스트 조합
    반환: str (Streamlit st.markdown 바로 사용 가능)
    """
    lines = []

    # ── 판정 헤더 ──────────────────────────────────────
    verdict_ko  = '🟢 실제 얼굴 (REAL)' if verdict == 'REAL' else '🔴 위조 공격 감지 (FAKE)'
    spoof_ko    = SPOOF_KO.get(spoof_type_idx, f'유형 {spoof_type_idx}')
    lines.append(f'**{verdict_ko}** — 신뢰도 {spoof_prob:.1%}')
    lines.append(f'**예측 공격 유형:** {spoof_ko}')

    if illum_idx is not None:
        lines.append(f'**조명:** {ILLUMINATION_NAMES.get(illum_idx, "알 수 없음")}')
    if env_idx is not None:
        lines.append(f'**환경:** {ENVIRONMENT_NAMES.get(env_idx, "알 수 없음")}')

    lines.append('')

    # ── Layer 1: Grad-CAM ──────────────────────────────
    lines.append('**[Layer 1 — Grad-CAM]** 어디를 봤는가')
    lines.append('> 히트맵: 이마·눈 주변 고활성 영역 (logit 기반, sigmoid 포화 해결)')
    lines.append('')

    # ── Layer 2: 수치 앵커링 ───────────────────────────
    lines.append('**[Layer 2 — 수치 앵커링]** 얼마나 강한 신호인가')
    lines.append(f'> Laplacian 분산: **{anchor_stats["laplacian"]}**'
                 f'  |  FFT 고주파: **{anchor_stats["fft_high"]}**')
    lines.append(f'> 해석: {anchor_interp}')

    # 공격 유형별 힌트 추가
    hint = SPOOF_EN_HINT.get(spoof_type_idx)
    if hint and verdict == 'FAKE':
        lines.append(f'> ✏️ {hint}')
    lines.append('')

    # ── Layer 3: LLaVA 캡션 ────────────────────────────
    lines.append('**[Layer 3 — VLM 자연어 분석]** 왜 그렇게 판단했는가')
    if llava_caption:
        lines.append(f'> {llava_caption}')
    else:
        lines.append('> *(해당 이미지 캡션 없음 — 배치 캡셔닝 결과 로드 필요)*')

    return '\n'.join(lines)


print('✅ Layer 3 (자연어 합성) 함수 준비 완료')

✅ Layer 3 (자연어 합성) 함수 준비 완료


## Cell 7 — 통합 `explain()` 함수

> 이미지 1장을 받아 3계층 XAI 결과를 딕셔너리로 반환.  
> Streamlit에서 `from xai_explainer import explain` 으로 바로 사용한다.

In [33]:
def explain(img_bgr, img_path=None, category=None,
            conv_layer=TARGET_LAYER, logit_layer=LOGIT_LAYER,
            spoof_layer=SPOOF_LAYER,
            heatmap_threshold=0.4,
            illum_idx=None, env_idx=None):
    """
    3계층 XAI 파이프라인 통합 함수

    Parameters
    ----------
    img_bgr : np.ndarray   BGR 이미지 (cv2.imread 결과)
    img_path: str | None   LLaVA 캡션 조회용 파일 경로 (없으면 캡션 생략)
    Returns
    -------
    dict with keys:
        verdict, spoof_prob, spoof_type_idx, spoof_type_name,
        heatmap_raw, heatmap_overlay,
        anchor_stats, anchor_interp,
        llava_caption, xai_text
    """
    inp = preprocess(img_bgr)

    # ── 모델 예측 ──────────────────────────────────────
    preds = model.predict(inp, verbose=0)
    # 멀티태스크: [binary_head, spoof_head] 또는 단일
    if isinstance(preds, list) and len(preds) >= 2:
        spoof_prob     = float(preds[0][0][0])
        spoof_type_raw = preds[1][0]
        spoof_type_idx = int(np.argmax(spoof_type_raw))  # 1-indexed (live=0 제외)
    else:
        spoof_prob     = float(preds[0][0]) if preds.ndim > 1 else float(preds[0])
        spoof_type_idx = 0

    verdict        = 'FAKE' if spoof_prob >= 0.5 else 'REAL'
    spoof_type_name= SPOOF_KO.get(spoof_type_idx, f'유형 {spoof_type_idx}')

    # ── Layer 1: Grad-CAM ──────────────────────────────
    heatmap_raw    = get_gradcam_logit(model, inp, conv_layer, logit_layer)
    heatmap_overlay= overlay_heatmap(img_bgr, heatmap_raw)

    # ── Layer 2: 수치 앵커링 ───────────────────────────
    cat_key = {1:'print', 2:'replay', 3:'mask'}.get(spoof_type_idx, 'live')
    # 히트맵 → 224×224 마스크
    hm_224  = cv2.resize(heatmap_raw, (224, 224))
    mask_224= hm_224 >= heatmap_threshold

    anchor_stats  = compute_pixel_features(img_bgr, mask_224 if mask_224.any() else None)
    anchor_interp = interpret_anchors(anchor_stats, category=cat_key)

    # ── Layer 3: LLaVA 캡션 + 자연어 합성 ──────────────
    llava_caption = get_llava_caption(img_path, category=category) if img_path else ''
    xai_text = build_xai_text(
        verdict, spoof_prob, spoof_type_idx,
        anchor_stats, anchor_interp,
        llava_caption, illum_idx, env_idx
    )

    return {
        'verdict'        : verdict,
        'spoof_prob'     : spoof_prob,
        'spoof_type_idx' : spoof_type_idx,
        'spoof_type_name': spoof_type_name,
        'heatmap_raw'    : heatmap_raw,
        'heatmap_overlay': heatmap_overlay,
        'anchor_stats'   : anchor_stats,
        'anchor_interp'  : anchor_interp,
        'llava_caption'  : llava_caption,
        'xai_text'       : xai_text,
    }


print('✅ explain() 함수 정의 완료')
print('  사용법: result = explain(img_bgr, img_path="path/to/image.jpg")')

✅ explain() 함수 정의 완료
  사용법: result = explain(img_bgr, img_path="path/to/image.jpg")


## Cell 8 — 단일 이미지 스모크 테스트

> 각 카테고리 1장씩 `explain()`을 실행해서 파이프라인 전체가 작동하는지 확인한다.

In [34]:
CATEGORIES = {
    'live'   : 'live',
    'print'  : 'print',
    'replay' : 'replay',
    'mask'   : 'mask',
}

smoke_results = {}

for cat, subdir in CATEGORIES.items():
    folder  = Path(CROP_DIR) / subdir
    img_paths = sorted(folder.glob('*.jpg'))[:1]
    if not img_paths:
        print(f'⚠️ {cat} 이미지 없음')
        continue

    p   = img_paths[0]
    img = cv2.imread(str(p))
    if img is None:
        print(f'⚠️ {p} 읽기 실패')
        continue

    r = explain(img, img_path=str(p), category=cat)
    smoke_results[cat] = r

    print(f'\n[{cat.upper()}] {p.name}')
    print(f'  판정: {r["verdict"]} ({r["spoof_prob"]:.1%})')
    print(f'  예측 유형: {r["spoof_type_name"]}')
    print(f'  Laplacian: {r["anchor_stats"]["laplacian"]}  |  FFT: {r["anchor_stats"]["fft_high"]}')
    print(f'  해석: {r["anchor_interp"]}')
    cap = r['llava_caption']
    print(f'  캡션: {cap[:80]}...' if len(cap) > 80 else f'  캡션: {cap or "(없음)"}')

print('\n✅ 스모크 테스트 완료')


[LIVE] 000369.jpg
  판정: REAL (20.0%)
  예측 유형: Live (실제 얼굴)
  Laplacian: 424.9  |  FFT: 475.4
  해석: 경계선 강조 (선명도 높음) / 고주파 에너지 부재 (압축/재촬영 흔적)
  캡션: The man's face is well-defined with a smooth complexion and visible pores. He ha...

[PRINT] 000072.jpg
  판정: FAKE (100.0%)
  예측 유형: Print Attack (인쇄 공격)
  Laplacian: 293.2  |  FFT: 1055.4
  해석: 선명도 보통 / 고주파 에너지 정상
  캡션: The face mask has a shiny appearance, which suggests that it might be made of a ...

[REPLAY] 000129.jpg
  판정: FAKE (100.0%)
  예측 유형: Replay Attack (화면 재촬영)
  Laplacian: 55.7  |  FFT: 600.2
  해석: 질감 평탄 (선명도 낮음) / 고주파 에너지 부재 (압축/재촬영 흔적)
  캡션: The image is a close-up of a man wearing a suit and tie. There are no noticeable...

[MASK] 000138.jpg
  판정: FAKE (100.0%)
  예측 유형: 3D Mask (입체 마스크)
  Laplacian: 153.3  |  FFT: 948.8
  해석: 질감 평탄 (선명도 낮음) / 고주파 에너지 부재 (압축/재촬영 흔적)
  캡션: The image features a woman's face with blue eyeshadow and earrings. There are no...

✅ 스모크 테스트 완료


In [31]:
# spoof head 출력 구조 확인
import numpy as np
from pathlib import Path
import cv2

img = cv2.imread(str(sorted(Path(f"{CROP_DIR}/live").glob("*.jpg"))[0]))
inp = preprocess(img)
preds = model.predict(inp, verbose=0)

print("preds 타입:", type(preds))
print("preds 길이:", len(preds))
print("preds[0] (binary):", preds[0])   # Real/Fake 확률
print("preds[1] (spoof) :", preds[1])   # 10종 분류
print("argmax:", np.argmax(preds[1][0]))

preds 타입: <class 'list'>
preds 길이: 2
preds[0] (binary): [[0.19953987]]
preds[1] (spoof) : [[6.4487499e-01 5.6959749e-03 3.4942797e-01 1.0871036e-06]]
argmax: 0


## Cell 9 — 통합 시각화 (4카테고리 × 3패널)

> 원본 / Grad-CAM 히트맵 / XAI 텍스트 요약을 나란히 출력한다.

In [39]:
import matplotlib.font_manager as fm
import matplotlib

# 폰트 직접 등록
font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
fm.fontManager.addfont(font_path)

# 폰트 이름 확인
prop = fm.FontProperties(fname=font_path)
print('폰트 이름:', prop.get_name())

# 적용
matplotlib.rcParams['font.family'] = prop.get_name()
matplotlib.rcParams['axes.unicode_minus'] = False

print('✅ 한글 폰트 적용 완료')

폰트 이름: NanumGothic
✅ 한글 폰트 적용 완료


In [ ]:
import matplotlib

# 폰트 없으면 영어로 대체
matplotlib.rcParams['axes.unicode_minus'] = False

fig = plt.figure(figsize=(16, 5 * len(smoke_results)))
fig.suptitle('Phase 4-E: 3-Layer XAI Integration Results',
             fontsize=15, fontweight='bold', y=1.01)

for row_i, (cat, r) in enumerate(smoke_results.items()):
    # ── 원본 이미지 ──────────────────────────────────────
    img_path = sorted(Path(f'{CROP_DIR}/{cat}').glob('*.jpg'))[0]
    img_bgr  = cv2.imread(str(img_path))
    img_rgb  = cv2.cvtColor(cv2.resize(img_bgr, (224, 224)), cv2.COLOR_BGR2RGB)

    ax1 = fig.add_subplot(len(smoke_results), 3, row_i * 3 + 1)
    ax1.imshow(img_rgb)
    ax1.set_title(f'[{cat.upper()}] Original', fontsize=11)
    ax1.axis('off')

    # ── Grad-CAM 히트맵 ──────────────────────────────────
    ax2 = fig.add_subplot(len(smoke_results), 3, row_i * 3 + 2)
    ax2.imshow(r['heatmap_overlay'])
    verdict_str = f'{r["verdict"]} ({r["spoof_prob"]:.1%})'
    color = 'red' if r['verdict'] == 'FAKE' else 'green'
    ax2.set_title(f'Layer 1: Grad-CAM\n{verdict_str}', fontsize=10, color=color)
    ax2.axis('off')

    # ── XAI 텍스트 요약 ──────────────────────────────────
    ax3 = fig.add_subplot(len(smoke_results), 3, row_i * 3 + 3)
    ax3.axis('off')
    summary = (
        f"Type: {r['spoof_type_name']}\n"
        f"Laplacian: {r['anchor_stats']['laplacian']}\n"
        f"FFT High:  {r['anchor_stats']['fft_high']}\n"
        f"Interp: {r['anchor_interp']}\n\n"
        f"Caption:\n{r['llava_caption'][:120] if r['llava_caption'] else '(none)'}"
    )
    ax3.text(0.05, 0.95, summary, transform=ax3.transAxes,
         fontsize=8, verticalalignment='top', fontfamily='NanumGothic',
             bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
    ax3.set_title('Layer 2+3: Anchoring + Caption', fontsize=10)

plt.tight_layout()
out_path = f'{REPORT_DIR}/11_xai_integration.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Saved: {out_path}')

## ✅ Phase 4-E 완료 체크리스트

| 항목 | 상태 |
|------|------|
| Drive 마운트 + 경로 설정 | ⬜ |
| 모델 로드 & 레이어 자동 감지 | ⬜ |
| LLaVA 캡션 DB 로드 | ⬜ |
| Layer 1: logit Grad-CAM 함수 | ⬜ |
| Layer 2: 수치 앵커링 (마스크 기반) | ⬜ |
| Layer 3: 자연어 설명 합성 | ⬜ |
| explain() 통합 함수 정의 | ⬜ |
| 4카테고리 스모크 테스트 | ⬜ |
| 통합 시각화 저장 (`11_xai_integration.png`) | ⬜ |
| `src/xai_explainer.py` 저장 | ⬜ |

---

### 다음 단계 — Phase 5: Streamlit 앱 구현
- `app/streamlit_app.py` 생성
- `from xai_explainer import explain` import
- 이미지 업로드 UI → `explain()` 호출 → 결과 표시
- Grad-CAM 히트맵 + 수치 + 자연어 설명 3단 레이아웃